# Fast3R 复现：从多视图图片到 3D 点云

这是一个面向学习和求职展示的、可重复运行的 Fast3R 实验记录。Fast3R 把多张同一场景的图片一次性送入 Transformer，预测每个视角的 global/local 3D 点图与置信度；相机位姿随后通过 PnP 从点图估计。

本 Notebook 分成两层：

1. **功能复现（主线）**：使用仓库提供的 Hugging Face checkpoint，完成视频抽帧、图片预处理、一次多视图推理、PnP 相机估计、置信度可视化和 PLY 点云导出。
2. **数据集评测**：公开 HF 权重可通过官方指标实现评测，DTU 22 场景已运行；本 Notebook 末尾进一步记录论文对应、median 指标差距、视角数与 local/global 点图消融。完整论文还包含其他数据集和独立训练实验。

> 建议环境：Python 3.11、CUDA GPU（建议显存至少 12 GB）和仓库根目录。CPU 可以完成环境检查和结果可视化，但不适合运行 ViT-L/512 的完整推理。

先运行下面的代码单元，确认 Notebook 使用的是 `fast3r` Conda 环境，并检查 Python、PyTorch、CUDA、GPU、Fast3R 和关键依赖。

## 0. 运行前准备

在终端执行一次下面的官方安装流程，然后在 Jupyter 中选择 `fast3r` kernel。已有环境可以直接跳过：

```bash
conda create -n fast3r python=3.11 cmake=3.14.0 -y
conda activate fast3r
# 按本机 CUDA 版本安装 PyTorch；再安装项目依赖
pip install -r requirements.txt
pip install -e .
python -m ipykernel install --user --name fast3r --display-name "fast3r"
```

仓库 README 特别提醒：不要安装 DUSt3R 的 cuROPE 模块，否则可能破坏 Fast3R 的预测。模型权重也可以提前下载到 `checkpoints/Fast3R_ViT_Large_512/`，这样 Notebook 不依赖运行时网络。

In [6]:
import os
import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path


def package_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "未安装"


print("=== Python 环境 ===")
print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("Conda 环境:", os.environ.get("CONDA_DEFAULT_ENV", "未检测到"))
print("系统:", platform.platform())

print("\n=== PyTorch / CUDA ===")
try:
    import torch

    print("PyTorch:", torch.__version__)
    print("Torch CUDA:", torch.version.cuda)
    print("CUDA 可用:", torch.cuda.is_available())
    print("GPU 数量:", torch.cuda.device_count())
    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()):
            properties = torch.cuda.get_device_properties(index)
            memory_gb = properties.total_memory / 1024**3
            print(f"GPU {index}: {properties.name} ({memory_gb:.1f} GB)")
except ImportError as error:
    print("PyTorch 导入失败:", error)

print("\n=== Fast3R / 关键依赖 ===")
for package_name in [
    "fast3r",
    "numpy",
    "open3d",
    "jupyterlab",
    "ipykernel",
    "gradio",
    "lightning",
    "huggingface-hub",
]:
    print(f"{package_name}: {package_version(package_name)}")

print("\n=== 项目路径 ===")
project_root = Path.cwd()
print("当前目录:", project_root)
for relative_path in [
    "fast3r",
    "checkpoints/Fast3R_ViT_Large_512",
    "demo_examples",
    "requirements.txt",
]:
    path = project_root / relative_path
    print(f"{relative_path}: {'存在' if path.exists() else '缺失'}")

print("\n=== Fast3R 导入检查 ===")
try:
    import fast3r
    from fast3r.dust3r.inference_multiview import inference
    from fast3r.models.fast3r import Fast3R

    print("Fast3R 导入: 成功")
    print("Fast3R 路径:", fast3r.__file__)
except Exception as error:
    print("Fast3R 导入失败:", repr(error))

=== Python 环境 ===
Python: 3.11.16
解释器: /home/yyz/miniconda3/envs/fast3r/bin/python
Conda 环境: fast3r
系统: Linux-6.8.0-138-generic-x86_64-with-glibc2.35

=== PyTorch / CUDA ===
PyTorch: 2.7.1+cu128
Torch CUDA: 12.8
CUDA 可用: True
GPU 数量: 1
GPU 0: NVIDIA GeForce RTX 5070 Ti Laptop GPU (11.5 GB)

=== Fast3R / 关键依赖 ===
fast3r: 1.0
numpy: 1.26.4
open3d: 0.19.0
jupyterlab: 4.6.3
ipykernel: 7.3.0
gradio: 5.23.0
lightning: 2.6.5
huggingface-hub: 1.30.0

=== 项目路径 ===
当前目录: /home/yyz/fast3r
fast3r: 存在
checkpoints/Fast3R_ViT_Large_512: 存在
demo_examples: 存在
requirements.txt: 存在

=== Fast3R 导入检查 ===
Fast3R 导入: 成功
Fast3R 路径: /home/yyz/fast3r/fast3r/__init__.py


## 在 Notebook 中启动 Gradio Demo

运行下面的代码单元后，模型和 Gradio 服务会在后台启动。请在浏览器中打开输出的本地地址完成上传和可视化。

> 说明：官方 Demo 还会启动 Viser 3D 服务并申请临时 share URL；如果网络受限，Gradio 页面可能能打开但 3D iframe 不可用。主线推理和本 Notebook 的 Plotly/PLY 预览不依赖这个临时链接。

In [7]:
import os
import subprocess
import sys
import time
from pathlib import Path

project_root = Path.cwd()
checkpoint_dir = project_root / "checkpoints" / "Fast3R_ViT_Large_512"
demo_script = project_root / "fast3r" / "viz" / "demo.py"

demo_process = globals().get("demo_process")
if demo_process is not None and demo_process.poll() is None:
    print(f"Demo 已在运行，PID: {demo_process.pid}")
    print("浏览器地址: http://127.0.0.1:7860")
else:
    import torch

    can_start = True
    if torch.cuda.is_available():
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        free_gb = free_bytes / 1024**3
        print(f"GPU 可用显存: {free_gb:.2f} / {total_bytes / 1024**3:.2f} GB")
        if free_gb < 4.0:
            can_start = False
            print("暂不启动 Demo：可用显存不足 4 GB。")
            print("请先停止其他 GPU 训练任务，再重新运行本单元。")

    if can_start:
        if not checkpoint_dir.exists():
            raise FileNotFoundError(f"找不到模型目录: {checkpoint_dir}")
        if not demo_script.exists():
            raise FileNotFoundError(f"找不到 Demo 脚本: {demo_script}")

        environment = os.environ.copy()
        environment["GRADIO_TEMP_DIR"] = str(project_root / "gradio_tmp_dir")
        demo_process = subprocess.Popen(
            [
                sys.executable,
                str(demo_script),
                "--checkpoint_dir",
                str(checkpoint_dir),
                "--examples_dir",
                str(project_root / "demo_examples"),
                "--output_dir",
                str(project_root / "demo_outputs"),
            ],
            cwd=project_root,
            env=environment,
            start_new_session=True,
        )
        time.sleep(2)
        if demo_process.poll() is not None:
            raise RuntimeError(f"Demo 启动失败，退出码: {demo_process.returncode}")

        print(f"Demo 已在后台启动，PID: {demo_process.pid}")
        print("请在浏览器打开: http://127.0.0.1:7860")
        print("如果端口被占用，请先运行停止单元，再重新启动。")

GPU 可用显存: 10.42 / 11.47 GB
Demo 已在后台启动，PID: 85761
请在浏览器打开: http://127.0.0.1:7860
如果端口被占用，请先运行停止单元，再重新启动。


## 停止 Notebook 启动的 Demo

在浏览器中完成观察后，运行下面的单元释放模型占用的显存。

In [8]:
demo_process = globals().get("demo_process")
if demo_process is not None and demo_process.poll() is None:
    demo_process.terminate()
    try:
        demo_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        demo_process.kill()
        demo_process.wait()
    print(f"Demo 已停止，PID: {demo_process.pid}")
    demo_process = None
else:
    print("当前没有由本 Notebook 启动的 Demo 进程。")

Demo 已停止，PID: 85761


## 直接运行多视图推理

下面的单元使用本地 Fast3R checkpoint 和仓库中的示例图片完成一次推理。先运行配置和图片加载单元，再运行模型推理单元。

In [9]:
from pathlib import Path

import torch

from fast3r.dust3r.inference_multiview import inference
from fast3r.dust3r.utils.image import load_images
from fast3r.utils.checkpoint_utils import load_model
from fast3r.viz.video_utils import extract_frames_from_video

project_root = Path.cwd()
checkpoint_dir = project_root / "checkpoints" / "Fast3R_ViT_Large_512"
output_dir = project_root / "demo_outputs" / "notebook_inference"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not checkpoint_dir.exists():
    raise FileNotFoundError(f"找不到模型目录: {checkpoint_dir}")

candidate_dirs = [
    project_root / "demo_outputs" / "notebook_family_frames",
    project_root / "demo_outputs" / "example_scenes",
]
image_extensions = {".jpg", ".jpeg", ".png", ".heic", ".heif"}
filelist = []
input_dir = None
for candidate_dir in candidate_dirs:
    candidate_files = sorted(
        path
        for path in candidate_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in image_extensions
    )
    if len(candidate_files) >= 2:
        input_dir = candidate_dir
        filelist = candidate_files
        break

if len(filelist) < 2:
    # 让 Notebook 从仓库自带的视频自动生成输入，而不是要求手动准备图片。
    video_path = project_root / "demo_examples" / "family" / "Family.mp4"
    input_dir = project_root / "demo_outputs" / "notebook_family_frames"
    input_dir.mkdir(parents=True, exist_ok=True)
    print("没有找到现成图片，正在从示例视频按约 1 FPS 抽帧：", video_path)
    extract_frames_from_video(str(video_path), str(input_dir))
    filelist = sorted(
        path for path in input_dir.iterdir()
        if path.is_file() and path.suffix.lower() in image_extensions
    )

if len(filelist) < 2:
    raise RuntimeError("至少需要 2 张输入图片才能进行多视图推理。")

if device.type != "cuda":
    print("警告：当前没有 CUDA GPU；准备步骤可以继续，但 ViT-L/512 推理建议切换到 GPU 内核。")
print("设备:", device)
print("模型目录:", checkpoint_dir)
print("输入目录:", input_dir)
print("示例图片数量:", len(filelist))
print("前 5 张图片:")
for path in filelist[:5]:
    print(" -", path.relative_to(project_root))

/home/yyz/miniconda3/envs/fast3r/lib/python3.11/site-packages/pl_bolts/__init__.py:11: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(numpy, tp_name):
/home/yyz/miniconda3/envs/fast3r/lib/python3.11/site-packages/pl_bolts/__init__.py:11: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  if not hasattr(numpy, tp_name):
/home/yyz/miniconda3/envs/fast3r/lib/python3.11/site-packages/lightning_fabric/__init__.py:29: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
/home/yyz/miniconda3/envs/fast3r/lib/python3.11/site-packages/pl_bolts/models/self_supervised/amdim/amdim_module.py:34: UnderReviewWarning: The feature generate_power_seq is currently marked under review. The compatibility with other Lightning projects

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
设备: cuda
模型目录: /home/yyz/fast3r/checkpoints/Fast3R_ViT_Large_512
输入目录: /home/yyz/fast3r/demo_outputs/notebook_family_frames
示例图片数量: 29
前 5 张图片:
 - demo_outputs/notebook_family_frames/frame_000000.jpg
 - demo_outputs/notebook_family_frames/frame_000001.jpg
 - demo_outputs/notebook_family_frames/frame_000002.jpg
 - demo_outputs/notebook_family_frames/frame_000003.jpg
 - demo_outputs/notebook_family_frames/frame_000004.jpg


In [10]:
max_views = 12
selected_filelist = [str(path) for path in filelist[:max_views]]
images = load_images(selected_filelist, size=512, verbose=True)

print(f"已加载 {len(images)} 个视角")
print("首个视角字段:", sorted(images[0].keys()))
print("首个视角图像形状:", tuple(images[0]["img"].shape))

>> Loading a list of 12 images
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000000.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000001.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000002.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000003.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000004.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000005.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000006.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000007.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/n

In [11]:
ALLOW_CPU_INFERENCE = False
if device.type != "cuda" and not ALLOW_CPU_INFERENCE:
    raise RuntimeError("Fast3R ViT-L/512 主线推理建议使用 CUDA GPU；如需 CPU 试跑，请先把 ALLOW_CPU_INFERENCE 改为 True 并把 max_views 调小。")

print("正在加载 Fast3R 模型...")
model, lit_module = load_model(
    str(checkpoint_dir),
    device=device,
    is_lightning_checkpoint=False,
)
model.eval()
lit_module.eval()

# inference_multiview 内部把 "16-mixed" / "32" 作为精度开关。
# profiling=True 会调用 CUDA synchronize，因此 CPU 时必须关闭 profiling。
inference_precision = "16-mixed" if device.type == "cuda" else "32"
profiling_enabled = device.type == "cuda"

with torch.inference_mode():
    inference_result = inference(
        images,
        model,
        device,
        dtype=inference_precision,
        verbose=True,
        profiling=profiling_enabled,
    )
output_dict, profiling_info = (
    inference_result if profiling_enabled else (inference_result, None)
)

print("推理完成")
print("推理精度:", inference_precision)
print("模型前向耗时:", profiling_info.get("total_time", "未提供") if profiling_info else "CPU 未启用 profiling")
print("输出字段:", sorted(output_dict.keys()))
print("视角数量:", len(output_dict["views"]))
print("预测数量:", len(output_dict["preds"]))

正在加载 Fast3R 模型...
Loading weights from local directory
>> Inference with model on 12 images
encode_images time: 0.5055067539215088
pos emb time: 0.0033676624298095703
decoder time: 0.3022279739379883
head prepare input time: 0.0012867450714111328
head forward time: 0.38594746589660645
total Fast3R forward time: 1.1985869407653809
推理完成
模型前向耗时: 1.1985869407653809
输出字段: ['loss', 'preds', 'views']
视角数量: 12
预测数量: 12


In [12]:
print("=== 预测结构 ===")
for index, prediction in enumerate(output_dict["preds"][:3]):
    print(f"视角 {index}:")
    print("  字段:", sorted(prediction.keys()))
    for key in ["pts3d_in_other_view", "conf", "pts3d_local", "conf_local"]:
        value = prediction.get(key)
        if value is not None:
            print(f"  {key}: shape={tuple(value.shape)}, dtype={value.dtype}")

first_points = output_dict["preds"][0]["pts3d_in_other_view"]
first_confidence = output_dict["preds"][0]["conf"]
print("首个视角点图数值范围:", float(first_points.min()), float(first_points.max()))
print("首个视角置信度范围:", float(first_confidence.min()), float(first_confidence.max()))
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"模型参数量: {parameter_count / 1e6:.1f}M")

=== 预测结构 ===
视角 0:
  字段: ['conf', 'conf_local', 'pts3d_in_other_view', 'pts3d_local']
  pts3d_in_other_view: shape=(1, 288, 512, 3), dtype=torch.float32
  conf: shape=(1, 288, 512), dtype=torch.float32
  pts3d_local: shape=(1, 288, 512, 3), dtype=torch.float32
  conf_local: shape=(1, 288, 512), dtype=torch.float32
视角 1:
  字段: ['conf', 'conf_local', 'pts3d_in_other_view', 'pts3d_local']
  pts3d_in_other_view: shape=(1, 288, 512, 3), dtype=torch.float32
  conf: shape=(1, 288, 512), dtype=torch.float32
  pts3d_local: shape=(1, 288, 512, 3), dtype=torch.float32
  conf_local: shape=(1, 288, 512), dtype=torch.float32
视角 2:
  字段: ['conf', 'conf_local', 'pts3d_in_other_view', 'pts3d_local']
  pts3d_in_other_view: shape=(1, 288, 512, 3), dtype=torch.float32
  conf: shape=(1, 288, 512), dtype=torch.float32
  pts3d_local: shape=(1, 288, 512, 3), dtype=torch.float32
  conf_local: shape=(1, 288, 512), dtype=torch.float32
首个视角点图数值范围: -0.6345456838607788 1.9926409721374512
首个视角置信度范围: 1.0 37.087173461

In [13]:
poses_c2w_batch, estimated_focals = lit_module.estimate_camera_poses(
    output_dict["preds"],
    niter_PnP=100,
    focal_length_estimation_method="first_view_from_global_head",
)

camera_poses = poses_c2w_batch[0]
focals = estimated_focals[0]
print("相机数量:", len(camera_poses))
print("第一个相机位姿形状:", tuple(camera_poses[0].shape))
print("估计焦距数量:", len(focals))
print("第一个相机位姿:\n", camera_poses[0])
print("估计焦距:", [float(focal) for focal in focals])

相机数量: 12
第一个相机位姿形状: (4, 4)
估计焦距数量: 12
第一个相机位姿:
 [[ 9.9999356e-01 -2.7541840e-03  2.2939586e-03  8.4927487e-05]
 [ 2.7572943e-03  9.9999529e-01 -1.3537960e-03  8.4198039e-04]
 [-2.2902193e-03  1.3601125e-03  9.9999654e-01  1.0126389e-04]
 [ 0.0000000e+00  0.0000000e+00  0.0000000e+00  1.0000000e+00]]
估计焦距: [311.0323486328125, 311.0323486328125, 311.0323486328125, 311.0323486328125, 311.0323486328125, 311.0323486328125, 311.0323486328125, 311.0323486328125, 311.0323486328125, 311.0323486328125, 311.0323486328125, 311.0323486328125]


In [14]:
import numpy as np

output_dir.mkdir(parents=True, exist_ok=True)
pointmaps = np.stack(
    [prediction["pts3d_in_other_view"].detach().cpu().numpy().squeeze(0)
     for prediction in output_dict["preds"]]
)
confidence_maps = np.stack(
    [prediction["conf"].detach().cpu().numpy().squeeze(0)
     for prediction in output_dict["preds"]]
)
poses_array = np.stack(
    [pose.detach().cpu().numpy() if hasattr(pose, "detach") else np.asarray(pose)
     for pose in camera_poses]
)
focals_array = np.asarray([float(focal) for focal in focals], dtype=np.float32)

result_path = output_dir / "reconstruction_results.npz"
np.savez_compressed(
    result_path,
    pointmaps=pointmaps,
    confidence_maps=confidence_maps,
    poses_c2w=poses_array,
    estimated_focals=focals_array,
    image_paths=np.asarray(selected_filelist),
)
print("复现结果已保存:", result_path)
print("点图数组:", pointmaps.shape)
print("置信度数组:", confidence_maps.shape)

复现结果已保存: /home/yyz/fast3r/demo_outputs/notebook_inference/reconstruction_results.npz
点图数组: (12, 288, 512, 3)
置信度数组: (12, 288, 512)


## 模块级验证

这一部分使用 forward hooks 观察一次真实推理中各组件的输出，不改变模型计算。每个组件都检查输出是否存在、形状是否合理、数值是否有限。

In [15]:
from collections.abc import Mapping


def tensor_summary(value):
    if torch.is_tensor(value):
        detached = value.detach()
        return {
            "type": "tensor",
            "shape": tuple(detached.shape),
            "dtype": str(detached.dtype),
            "device": str(detached.device),
            "finite": bool(torch.isfinite(detached).all().item()),
            "min": float(detached.min().item()) if detached.numel() else None,
            "max": float(detached.max().item()) if detached.numel() else None,
        }
    if isinstance(value, Mapping):
        return {str(key): tensor_summary(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [tensor_summary(item) for item in value[:3]]
    return {"type": type(value).__name__}


module_outputs = {}


def capture_output(module_name):
    def hook(_module, _inputs, output):
        module_outputs[module_name] = tensor_summary(output)
    return hook


hook_handles = [
    model.encoder.patch_embed.register_forward_hook(capture_output("patch_embed")),
    model.encoder.register_forward_hook(capture_output("encoder")),
    model.decoder.register_forward_hook(capture_output("decoder")),
    model.downstream_head.register_forward_hook(capture_output("global_head")),
]
if model.downstream_head_local is not None:
    hook_handles.append(
        model.downstream_head_local.register_forward_hook(capture_output("local_head"))
    )

hooked_result = inference(
    images,
    model,
    device,
    dtype=inference_precision,
    verbose=False,
    profiling=profiling_enabled,
)
hooked_output, hooked_profile = (
    hooked_result if profiling_enabled else (hooked_result, None)
)

for handle in hook_handles:
    handle.remove()

required_modules = {"patch_embed", "encoder", "decoder", "global_head"}
missing_modules = required_modules - module_outputs.keys()
if missing_modules:
    raise RuntimeError(f"没有捕获到模块输出: {sorted(missing_modules)}")

for module_name, summary in module_outputs.items():
    print(f"[{module_name}]\n{summary}")

hooked_predictions = hooked_output["preds"]
assert len(hooked_predictions) == len(images)
assert all("pts3d_in_other_view" in prediction for prediction in hooked_predictions)
assert all(torch.isfinite(prediction["pts3d_in_other_view"]).all() for prediction in hooked_predictions)
assert all(torch.isfinite(prediction["conf"]).all() for prediction in hooked_predictions)
print("模块级验证通过")

encode_images time: 0.2693517208099365
pos emb time: 0.0005466938018798828
decoder time: 0.29361510276794434
head prepare input time: 0.0002491474151611328
head forward time: 0.2619936466217041
total Fast3R forward time: 0.8259179592132568
[patch_embed]
[{'type': 'tensor', 'shape': (12, 576, 1024), 'dtype': 'torch.float16', 'device': 'cuda:0', 'finite': True, 'min': -13.984375, 'max': 13.3203125}, {'type': 'tensor', 'shape': (12, 576, 2), 'dtype': 'torch.int64', 'device': 'cuda:0', 'finite': True, 'min': 0.0, 'max': 31.0}]
[encoder]
[{'type': 'tensor', 'shape': (12, 576, 1024), 'dtype': 'torch.float32', 'device': 'cuda:0', 'finite': True, 'min': -8.948944091796875, 'max': 10.216102600097656}, {'type': 'tensor', 'shape': (12, 576, 2), 'dtype': 'torch.int64', 'device': 'cuda:0', 'finite': True, 'min': 0.0, 'max': 31.0}]
[decoder]
[{'type': 'tensor', 'shape': (1, 6912, 1024), 'dtype': 'torch.float32', 'device': 'cuda:0', 'finite': True, 'min': -8.948944091796875, 'max': 10.216102600097656

## 视角规模与性能验证

用相同场景的不同视角数量运行推理，记录前向时间和显存峰值。这个实验用于验证模型的多视图输入行为，不等同于论文完整 benchmark。

In [16]:
import gc
import time

benchmark_results = []
for view_count in [4, 8, 12]:
    benchmark_images = images[:view_count]
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
    start_time = time.perf_counter()
    benchmark_result = inference(
        benchmark_images,
        model,
        device,
        dtype=inference_precision,
        verbose=False,
        profiling=profiling_enabled,
    )
    benchmark_output, benchmark_profile = (
        benchmark_result if profiling_enabled else (benchmark_result, None)
    )
    elapsed = time.perf_counter() - start_time
    peak_memory_gb = (
        torch.cuda.max_memory_allocated(device) / 1024**3
        if device.type == "cuda"
        else float("nan")
    )
    benchmark_results.append(
        {
            "views": view_count,
            "total_time": (
                float(benchmark_profile["total_time"])
                if benchmark_profile is not None
                else float("nan")
            ),
            "wall_time": elapsed,
            "peak_memory_gb": peak_memory_gb,
            "output_count": len(benchmark_output["preds"]),
        }
    )
    del benchmark_images, benchmark_output, benchmark_profile
    gc.collect()

for result in benchmark_results:
    print(
        f"视角={result['views']:>2} | "
        f"模型时间={result['total_time']:.3f}s | "
        f"墙钟时间={result['wall_time']:.3f}s | "
        f"峰值显存={result['peak_memory_gb']:.2f}GB | "
        f"输出={result['output_count']}"
    )

assert all(result["output_count"] == result["views"] for result in benchmark_results)
print("视角规模验证通过")

encode_images time: 0.12180614471435547
pos emb time: 0.0004839897155761719
decoder time: 0.08476042747497559
head prepare input time: 0.0002491474151611328
head forward time: 0.07056307792663574
total Fast3R forward time: 0.27797818183898926
encode_images time: 0.15106964111328125
pos emb time: 0.0005750656127929688
decoder time: 0.16988062858581543
head prepare input time: 0.0001475811004638672
head forward time: 0.14459848403930664
total Fast3R forward time: 0.4664151668548584
encode_images time: 0.27707791328430176
pos emb time: 0.0007290840148925781
decoder time: 0.2924051284790039
head prepare input time: 0.00024580955505371094
head forward time: 0.30933308601379395
total Fast3R forward time: 0.8799545764923096
视角= 4 | 模型时间=0.278s | 墙钟时间=0.287s | 峰值显存=4.56GB | 输出=4
视角= 8 | 模型时间=0.466s | 墙钟时间=0.481s | 峰值显存=5.31GB | 输出=8
视角=12 | 模型时间=0.880s | 墙钟时间=0.900s | 峰值显存=6.13GB | 输出=12
视角规模验证通过


## 点云导出与结果检查

使用 global prediction 的点图和置信度生成彩色 PLY。置信度阈值采用全体点的百分位数，避免低置信度区域淹没可视化结果。

In [17]:
import numpy as np
import open3d as o3d

all_points = []
all_colors = []
all_confidences = []

for prediction, view in zip(output_dict["preds"], images):
    points = prediction["pts3d_in_other_view"].detach().cpu().numpy().squeeze(0)
    confidence = prediction["conf"].detach().cpu().numpy().squeeze(0)
    image_tensor = view["img"].detach().cpu().squeeze(0).permute(1, 2, 0).numpy()

    if image_tensor.min() < 0:
        image_tensor = (image_tensor + 1.0) / 2.0
    if image_tensor.max() > 1:
        image_tensor = image_tensor / 255.0
    colors = np.clip(image_tensor, 0.0, 1.0)

    all_points.append(points.reshape(-1, 3))
    all_colors.append(colors.reshape(-1, 3))
    all_confidences.append(confidence.reshape(-1))

all_points = np.concatenate(all_points, axis=0)
all_colors = np.concatenate(all_colors, axis=0)
all_confidences = np.concatenate(all_confidences, axis=0)

confidence_percentile = 45
confidence_threshold = np.percentile(all_confidences, confidence_percentile)
valid = (
    np.isfinite(all_points).all(axis=1)
    & np.isfinite(all_colors).all(axis=1)
    & np.isfinite(all_confidences)
    & (all_confidences >= confidence_threshold)
)

filtered_points = all_points[valid]
filtered_colors = all_colors[valid]
max_points = 1_000_000
if len(filtered_points) > max_points:
    sample_indices = np.linspace(0, len(filtered_points) - 1, max_points, dtype=int)
    filtered_points = filtered_points[sample_indices]
    filtered_colors = filtered_colors[sample_indices]

point_cloud = o3d.geometry.PointCloud()
point_cloud.points = o3d.utility.Vector3dVector(filtered_points.astype(np.float64))
point_cloud.colors = o3d.utility.Vector3dVector(filtered_colors.astype(np.float64))
ply_path = output_dir / "reconstruction_pointcloud.ply"
written = o3d.io.write_point_cloud(str(ply_path), point_cloud)

if not written:
    raise RuntimeError(f"PLY 写入失败: {ply_path}")
print("置信度阈值:", float(confidence_threshold))
print("过滤后点数:", len(filtered_points))
print("PLY 已保存:", ply_path)
assert ply_path.exists() and ply_path.stat().st_size > 0

置信度阈值: 9.571455001831055
过滤后点数: 973464
PLY 已保存: /home/yyz/fast3r/demo_outputs/notebook_inference/reconstruction_pointcloud.ply


## 复现结果汇总与论文级评估边界

到这里完成的是本地 checkpoint 的组件级和端到端推理验证。论文中的 DTU、7-Scenes、Neural-RGBD 等指标还需要按照项目文档准备预处理数据，并运行 `configs/eval/` 下对应的评估配置。

In [18]:
summary = {
    "environment": {
        "conda": os.environ.get("CONDA_DEFAULT_ENV", "unknown"),
        "device": str(device),
        "cuda": bool(torch.cuda.is_available()),
    },
    "input_views": len(images),
    "predictions": len(output_dict["preds"]),
    "module_hooks": sorted(module_outputs.keys()),
    "pose_count": len(camera_poses),
    "pointcloud_points": len(filtered_points),
    "results_file": str(result_path),
    "ply_file": str(ply_path),
}

print("=== Fast3R 复现汇总 ===")
print("环境检查:", summary["environment"])
print("输入/预测视角:", summary["input_views"], "/", summary["predictions"])
print("已捕获模块:", ", ".join(summary["module_hooks"]))
print("相机位姿数量:", summary["pose_count"])
print("PLY 点数:", summary["pointcloud_points"])
print("NPZ:", summary["results_file"])
print("PLY:", summary["ply_file"])

assert summary["input_views"] == summary["predictions"] == summary["pose_count"]
assert {"patch_embed", "encoder", "decoder", "global_head"}.issubset(
    set(summary["module_hooks"])
)
assert Path(summary["results_file"]).exists()
assert Path(summary["ply_file"]).exists()
print("本地组件级复现全部通过")

=== Fast3R 复现汇总 ===
环境检查: {'conda': 'fast3r', 'device': 'cuda', 'cuda': True}
输入/预测视角: 12 / 12
已捕获模块: decoder, encoder, global_head, local_head, patch_embed
相机位姿数量: 12
PLY 点数: 973464
NPZ: /home/yyz/fast3r/demo_outputs/notebook_inference/reconstruction_results.npz
PLY: /home/yyz/fast3r/demo_outputs/notebook_inference/reconstruction_pointcloud.ply
本地组件级复现全部通过


## DTU 代表性数据集评测

DTU 是第一个正式评测集。下面保留数据检查单元；下载默认关闭。已有的22场景结果见末尾的论文核对单元；公开 HF 权重通过 `scripts/fast3r_hf_dtu_eval.py` 使用官方指标实现评测。

In [24]:
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

project_root = Path.cwd()
data_root = project_root / "data"
dtu_root = data_root / "dtu_test_mvsnet_release"
dtu_drive_url = "https://drive.google.com/drive/folders/1bqtcVf8lK4VC8LgG-SIGRBECcrFqM7Wy"
ALLOW_DATA_DOWNLOAD = False  # 改为 True 才会在 Notebook 中安装 gdown 并下载大文件

free_gb = shutil.disk_usage(project_root).free / 1024**3
print(f"当前磁盘可用空间: {free_gb:.1f} GB")
print("DTU 目标目录:", dtu_root)

if dtu_root.exists() and any(dtu_root.iterdir()):
    entries = [path for path in dtu_root.iterdir()]
    print(f"DTU 目录已存在，包含 {len(entries)} 个顶层条目。")
else:
    print("DTU 预处理数据尚未下载。")
    print("官方预处理数据来源:", dtu_drive_url)
    if not ALLOW_DATA_DOWNLOAD:
        print("为避免误触发大文件下载，默认不自动下载。请按 README 手动准备数据，或把 ALLOW_DATA_DOWNLOAD 改为 True。")
    elif free_gb < 12:
        print("警告：可用空间低于 12 GB，建议先清理磁盘后再下载。")
    else:
        try:
            import gdown
        except ImportError:
            print("正在将 gdown 安装到当前 Notebook 内核...")
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "gdown"],
                cwd=project_root,
                check=True,
            )
            importlib.invalidate_caches()
            import gdown

        data_root.mkdir(parents=True, exist_ok=True)
        print("开始下载 DTU 预处理数据，这可能需要较长时间。")
        try:
            subprocess.run(
                [
                    sys.executable,
                    "-m",
                    "gdown",
                    "--folder",
                    dtu_drive_url,
                    "-O",
                    str(dtu_root),
                ],
                cwd=project_root,
                check=True,
            )
        except subprocess.CalledProcessError as error:
            print("DTU 下载失败，退出码:", error.returncode)
            print("可能原因是当前网络无法访问 Google Drive。")
            print("请在可访问该链接的网络中手动下载并解压到:", dtu_root)
        else:
            print("DTU 下载完成。")

当前磁盘可用空间: 9.0 GB
DTU 目标目录: /home/yyz/fast3r/data/dtu_test_mvsnet_release
DTU 目录已存在，包含 22 个顶层条目。


### 运行 DTU 评测

下面保留原始 Lightning 入口检查；缺少 last.ckpt 时该入口不运行。公开 HF 权重同样可以评测：使用 `scripts/fast3r_hf_dtu_eval.py`；Notebook 末尾读取已完成的结果。文件格式本身不能证明某项指标无法复现。

In [25]:
import shlex
import subprocess
from pathlib import Path

project_root = Path.cwd()
dtu_root = project_root / "data" / "dtu_test_mvsnet_release"
lightning_checkpoint = project_root / "checkpoints" / "last.ckpt"

if not dtu_root.exists():
    print("找不到 DTU 数据目录，请先运行上一个下载单元。")
elif not lightning_checkpoint.exists():
    print("找不到 Lightning checkpoint:", lightning_checkpoint)
    print("当前的 checkpoints/Fast3R_ViT_Large_512/model.safetensors 不能直接用于 fast3r/eval.py。")
    print("请将训练得到的 last.ckpt 放到上述路径，或修改 lightning_checkpoint 变量。")
else:
    dtu_dataset = (
        "[DTU(split='test', "
        f"ROOT='{dtu_root}', resolution=512, num_seq=1, "
        "full_video=True, kf_every=5)]"
    )
    command = [
        sys.executable,
        "fast3r/eval.py",
        "eval=ablation_recon_better_inference_hp/ablation_recon_better_inference_hp",
        f"ckpt_path={lightning_checkpoint}",
        "slurm_job_id=local",
        f"data.data_root={data_root}",
        f"data.data_module.validation_datasets={dtu_dataset}",
        "trainer.limit_val_batches=1",
    ]
    print("即将执行:")
    print(" ".join(shlex.quote(part) for part in command))
    subprocess.run(command, cwd=project_root, check=True)

找不到 Lightning checkpoint: /home/yyz/fast3r/checkpoints/last.ckpt
当前的 checkpoints/Fast3R_ViT_Large_512/model.safetensors 不能直接用于 fast3r/eval.py。
请将训练得到的 last.ckpt 放到上述路径，或修改 lightning_checkpoint 变量。


## 教学可视化：先看输入，再看模型信心

把原始图片和模型输出并排画出来，有助于理解一个关键事实：Fast3R 的输出不是一张普通深度图，而是每个像素对应的三维坐标 `(x, y, z)` 以及一个置信度。

In [ ]:
if "images" not in globals():
    print("尚未加载图片，请先运行前面的图片加载单元。")
else:
    import matplotlib.pyplot as plt
    from fast3r.dust3r.utils.image import rgb

    gallery_count = min(6, len(images))
    fig, axes = plt.subplots(2, 3, figsize=(15, 7))
    for axis, view, path in zip(axes.flat, images[:gallery_count], selected_filelist[:gallery_count]):
        axis.imshow(rgb(view["img"][0]))
        axis.set_title(Path(path).name)
        axis.axis("off")
    for axis in axes.flat[gallery_count:]:
        axis.axis("off")
    fig.suptitle("输入的多视图图片")
    fig.tight_layout()
    plt.show()

In [ ]:
if "output_dict" not in globals():
    print("尚未完成推理，请先运行模型推理单元。")
else:
    import matplotlib.pyplot as plt
    from fast3r.dust3r.utils.image import rgb

    show_count = min(4, len(output_dict["preds"]))
    fig, axes = plt.subplots(show_count, 2, figsize=(9, 3 * show_count), squeeze=False)
    for row in range(show_count):
        image = rgb(images[row]["img"][0])
        confidence = output_dict["preds"][row]["conf"].detach().cpu().squeeze().numpy()
        low, high = np.percentile(confidence, [5, 95])
        axes[row, 0].imshow(image)
        axes[row, 0].set_title(f"view {row}: RGB")
        axes[row, 1].imshow(confidence, cmap="turbo", vmin=low, vmax=high)
        axes[row, 1].set_title(f"view {row}: confidence")
        axes[row, 0].axis("off")
        axes[row, 1].axis("off")
    fig.tight_layout()
    plt.show()

## 相机轨迹：从 3D 点图估计 camera-to-world 位姿

`estimate_camera_poses` 使用 Fast3R 的点图和 fast-PnP 估计每个视角的相机到世界变换。下面只画平移部分，因此它是一个直观的轨迹检查，不是带尺度对齐的相机姿态 benchmark。

In [ ]:
if "camera_poses" not in globals():
    print("尚未估计相机位姿，请先运行 PnP 单元。")
else:
    import matplotlib.pyplot as plt

    camera_centers = np.stack([np.asarray(pose)[:3, 3] for pose in camera_poses])
    fig = plt.figure(figsize=(8, 6))
    axis = fig.add_subplot(111, projection="3d")
    axis.plot(camera_centers[:, 0], camera_centers[:, 1], camera_centers[:, 2], "o-", linewidth=2)
    for index, center in enumerate(camera_centers):
        axis.text(*center, str(index), fontsize=8)
    axis.set_xlabel("x")
    axis.set_ylabel("y")
    axis.set_zlabel("z")
    axis.set_title("estimated camera trajectory")
    plt.tight_layout()
    plt.show()

## 点云预览：浏览器内查看稀疏采样

PLY 是给 Open3D / MeshLab 使用的持久化产物；这里用 Plotly 抽样最多 20,000 个点，避免 Notebook 输出过大。

In [ ]:
if "filtered_points" not in globals():
    print("尚未生成点云，请先运行 PLY 导出单元。")
else:
    import plotly.graph_objects as go

    rng = np.random.default_rng(7)
    preview_count = min(20_000, len(filtered_points))
    preview_indices = rng.choice(len(filtered_points), size=preview_count, replace=False)
    preview_points = filtered_points[preview_indices]
    preview_colors = (filtered_colors[preview_indices] * 255).astype(np.uint8)
    color_strings = [f"rgb({r},{g},{b})" for r, g, b in preview_colors]
    figure = go.Figure(go.Scatter3d(
        x=preview_points[:, 0], y=preview_points[:, 1], z=preview_points[:, 2],
        mode="markers", marker={"size": 2, "color": color_strings, "opacity": 0.75},
    ))
    figure.update_layout(title=f"Fast3R point cloud preview ({preview_count:,} points)", height=650,
                          scene_aspectmode="data")
    figure.show()

## 视角规模实验图

这个小实验记录的是本机前向耗时和显存随视角数的变化。它能支撑‘一次前向处理多视图’的工程观察，但不能替代论文中的完整数据集指标。

In [ ]:
if not globals().get("benchmark_results"):
    print("尚未运行视角规模实验。")
else:
    import matplotlib.pyplot as plt
    import pandas as pd
    from IPython.display import display

    benchmark_table = pd.DataFrame(benchmark_results)
    display(benchmark_table)
    axis = benchmark_table.plot(x="views", y="wall_time", marker="o", figsize=(7, 4), legend=False)
    axis.set_ylabel("wall time (s)")
    axis.set_title("inference time vs. number of views")
    plt.tight_layout()
    plt.show()

## 生成可交付的复现卡片

把 checkpoint、输入视角数、推理精度、输出文件和本地 benchmark 一起保存，面试时比只给一张截图更容易说明实验是如何复现的。

In [ ]:
import json
from datetime import datetime, timezone

if "output_dir" not in globals():
    print("尚未完成主线推理，暂不生成复现卡片。")
else:
    repro_card = {
        "project": "Fast3R",
        "purpose": "functional reproduction of multi-view inference",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "checkpoint": str(checkpoint_dir),
        "input_views": len(images),
        "image_size": list(images[0]["img"].shape[-2:]),
        "device": str(device),
        "precision": inference_precision,
        "forward_time_seconds": (float(profiling_info["total_time"]) if profiling_info else None),
        "point_count_after_confidence_filter": int(len(filtered_points)),
        "artifacts": {"npz": str(result_path), "ply": str(ply_path)},
        "benchmark": benchmark_results,
        "scope_note": "This card describes functional inference. Dataset experiments and paper comparisons are recorded separately in the paper audit cells.",
    }
    repro_card_path = output_dir / "reproducibility_card.json"
    repro_card_path.write_text(json.dumps(repro_card, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(repro_card, ensure_ascii=False, indent=2))
    print("复现卡片已保存:", repro_card_path)

## 面试 / 简历表述模板

完成主线后，可以用下面这段作为简历初稿，再把自己的实际硬件和复现卡片中的数字替换进去：

- **Fast3R 多视图 3D 重建复现（PyTorch / CUDA）**：基于官方 ViT-L/512 checkpoint，搭建从视频抽帧、图像归一化、多视图一次前向到 PnP 相机估计的端到端推理流程；导出带 RGB 与置信度筛选的 `.ply` 点云和 `.npz` 结果。
- 设计模块级 hook 与视角规模实验，检查 patch embedding、encoder、decoder、global/local head 的张量形状和有限性，并记录 4/8/12 视角的前向时间与显存。

面试时应说明实际覆盖范围：公开权重推理、DTU 22 场景评测，以及后面的推理视角数和点图头消融。论文数值存在差距，其余数据集和重新训练实验未完成。

## 论文逐项核对：复现到了哪里？

对应 [Fast3R arXiv v2](https://arxiv.org/html/2501.13928v2)，完整清单见 [PAPER_REPRODUCTION.md](PAPER_REPRODUCTION.md)。以下单元可独立运行，直接读取已保存的实测 JSON，无需 GPU。

| 步骤 | 论文位置 | 本项目验证 |
| --- | --- | --- |
| 1 | §3.1、§3.3、Figure 2 | 共享 encoder、全视角融合 Transformer、global/local 点图与置信度；上面的 hook 已运行 |
| 2 | §3.2，Eq. (1)–(3) | 阅读归一化回归和置信度损失；未重新训练 |
| 3 | §4.3、Table 4 | 22 场景 DTU，比较每场景 median 的均值 |
| 4 | §5.4、Table 5 | 同一次预测的 aligned local 与 global 对比 |
| 5 | §5.1、Figure 5 | 3/5/10/20 视角的均匀采样适配实验 |
| 6 | §4.1、Table 2 | 本机预热后重复计时、allocated/reserved 显存 |

完整复现仍缺 §4.2/Table 1、§4.3/Table 3、训练消融和附录实验。本机结果与论文环境不同，不能将脚本运行成功当成所有结论验证成功。

### 步骤 2：从论文公式找到代码

每个像素输出一个 3D 点。归一化回归先把预测点和 GT 点分别除以各自点到原点距离的平均值，再计算 L2 点误差，这样可减少整体尺度差异的影响。置信度损失让模型对可靠点赋予更高权重。

打开 `configs/model/fast3r.yaml`：训练配置连接 `ConfLossMultiviewV2(Regr3DMultiviewV4(L21Loss))`，alpha=0.2。实现位于 `fast3r/dust3r/losses.py`。代码使用 `conf * loss - alpha * log(conf)`，同时归一化 global/local 项；阅读论文 Eq. (3) 时注意 HTML 中 log 项符号与实际代码不同，以已发布实现为实验依据。合成点图的数值与梯度检查通过，但尚未从头训练。

`fast3r/models/fast3r.py` 中 `_get_random_image_pos` 固定第一视角为参考坐标系，对其他视角随机采样 image-index embedding。因此新实验必须保存 seed 与输入标签。

In [1]:
import json
from pathlib import Path
loss_checks = json.loads(Path('results/loss_checks.json').read_text())
assert loss_checks['status'] == 'passed' and loss_checks['finite_gradients']
print(json.dumps(loss_checks, indent=2, ensure_ascii=False))
print('这里是损失数值/反向传播验证，没有进行模型训练。')

{
  "paper_section": "3.2, Equations 1-3",
  "status": "passed",
  "scope": "Synthetic CPU loss/backprop invariants only; no model training or dataset benchmark.",
  "seed": 42,
  "views": 3,
  "confidence_alpha": 0.2,
  "exact_prediction_loss": -0.13862943649291992,
  "uniformly_scaled_prediction_loss": -0.13862931728363037,
  "perturbed_prediction_loss": -0.05105750262737274,
  "finite_gradients": true,
  "confidence_sign": "released implementation: conf * regression - alpha * log(conf)"
}
这里是损失数值/反向传播验证，没有进行模型训练。


In [2]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt

paper_root = Path.cwd()
assert (paper_root / 'README.md').exists(), '请在仓库根目录运行'
dtu_history = json.loads((paper_root / 'demo_outputs/paper_eval/dtu_all.json').read_text())
assert dtu_history['scene_count'] == len(dtu_history['scenes']) == 22
assert len({r['scene'] for r in dtu_history['scenes']}) == 22
for metric, value in dtu_history['aggregate_mean'].items():
    recomputed = np.mean([r['metrics'][metric] for r in dtu_history['scenes']])
    assert np.isfinite(value) and np.isclose(value, recomputed), metric
print('历史 DTU：22 场景完整，所有 aggregate_mean 与逐场景均值一致')
dtu_seeded = json.loads((paper_root / 'results/dtu_seed42_stride5.json').read_text())
assert dtu_seeded['scene_count'] == len(dtu_seeded['scenes']) == 22
assert dtu_seeded['seed'] == 42 and dtu_seeded['kf_every'] == 5
assert dtu_seeded['head_chunk_size'] == 2
for metric, value in dtu_seeded['aggregate_mean'].items():
    assert np.isclose(value, np.mean([r['metrics'][metric] for r in dtu_seeded['scenes']]))
comparison = pd.DataFrame([
    {'指标': 'Accuracy median', '本地历史运行': dtu_history['aggregate_mean']['accuracy_median'], '论文 Table 4': 1.706},
    {'指标': 'Completion median', '本地历史运行': dtu_history['aggregate_mean']['completion_median'], '论文 Table 4': 0.857},
])
comparison['seed=42 stride=5'] = [dtu_seeded['aggregate_mean']['accuracy_median'], dtu_seeded['aggregate_mean']['completion_median']]
comparison['新运行相对论文高出 (%)'] = (comparison['seed=42 stride=5'] / comparison['论文 Table 4'] - 1) * 100
display(comparison.round(4))
print('这两个指标越低越好；seeded 运行仍有数值差距。历史报告未记录 seed；新运行同时改了seed、遍历顺序与DPT分块，不能把差异全归因于seed。')
figure_dir = paper_root / 'results/figures'
figure_dir.mkdir(parents=True, exist_ok=True)

历史 DTU：22 场景完整，所有 aggregate_mean 与逐场景均值一致


,指标,本地历史运行,论文 Table 4,seed=42 stride=5,新运行相对论文高出 (%)
0,Accuracy median,2.9306,1.706,2.0827,22.0792
1,Completion median,1.8200,0.857,1.0311,20.3142


这两个指标越低越好；seeded 运行仍有数值差距。历史报告未记录 seed；新运行同时改了seed、遍历顺序与DPT分块，不能把差异全归因于seed。


### 步骤 4–5：消融控制变量

运行 `python scripts/fast3r_paper_experiments.py --output-json results/dtu_paper_experiments.json` 生成新实验。每次结果写入前先保存临时文件，再原子替换，避免把写了一半的 JSON 当成完整结果。

local/global 分支复用 10 视角的同一次预测。改变的只有用于指标的点图和相应置信度；两种分支都实际计算了两个 head，所以其比较不等于测量移除 local head 的速度。

视角数实验均匀抽取完整 49 帧中的 N 张，保留第一帧，属于 Figure 5 的本机适配。不同 N 也会改变用于评价的 GT 点集；它与官方 stride 采样不同。若观测到某指标没有单调改善，应如实报告。

In [3]:
ablation = json.loads((paper_root / 'results/dtu_paper_experiments.json').read_text())
assert ablation['status'] == 'completed', '等待实验完成；如有 OOM 请先检查报告'
expected = len(ablation['scenes_expected']) * len(ablation['view_counts_expected'])
assert len(ablation['rows']) == expected == 88
assert len({(r['scene'], r['view_count']) for r in ablation['rows']}) == expected
assert all(r['status'] == 'completed' for r in ablation['rows'])
records = []
for r in ablation['rows']:
    assert len(r['input_labels']) == r['view_count']
    for head, metrics in r['metrics'].items():
        assert all(np.isfinite(v) for v in metrics.values())
        records.append({'scene': r['scene'], 'views': r['view_count'], 'head': head, **metrics})
ablation_df = pd.DataFrame(records)
ablation_means = ablation_df.groupby(['views', 'head']).mean(numeric_only=True)
display(ablation_means.round(4))
for (views, head), row in ablation_means.iterrows():
    stored = ablation['aggregate'][f'{views}_views_{head}']
    assert stored['scene_count'] == 22
    for k, v in stored['aggregate_mean'].items():
        assert np.isclose(v, row[k]), (views, head, k)
paired = ablation_means.loc[10]
display(pd.DataFrame({'local': paired.loc['local'], 'global': paired.loc['global'],
                      'global - local': paired.loc['global'] - paired.loc['local']}).round(4))
print('全部 88 次前向与 22 组配对 head 评测通过；差值>0表示global距离误差更大。')

accuracy  accuracy_median  completion  completion_median  \
views head                                                               
3     local     6.3791           3.3735      5.6077             2.9197   
5     local     4.8601           2.6129      3.5799             1.8621   
10    global    4.3514           2.0541      2.7941             1.0239   
      local     4.1451           1.9145      2.3889             0.9070   
20    local     4.0098           1.7020      2.5146             0.9653   

                 nc1  nc1_median     nc2  nc2_median  
views head                                            
3     local   0.7058      0.7894  0.6841      0.7638  
5     local   0.6944      0.7771  0.6675      0.7411  
10    global  0.6600      0.7384  0.6283      0.6930  
      local   0.6769      0.7588  0.6432      0.7124  
20    local   0.6415      0.7152  0.6071      0.6643

,local,global,global - local
accuracy,4.1451,4.3514,0.2063
accuracy_median,1.9145,2.0541,0.1397
completion,2.3889,2.7941,0.4052
completion_median,0.9070,1.0239,0.1169
nc1,0.6769,0.6600,-0.0169
nc1_median,0.7588,0.7384,-0.0204
nc2,0.6432,0.6283,-0.0150
nc2_median,0.7124,0.6930,-0.0194


全部 88 次前向与 22 组配对 head 评测通过；差值>0表示global距离误差更大。


In [4]:
local_means = ablation_means.xs('local', level='head')
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, metric in zip(axes, ['accuracy_median', 'completion_median']):
    ax.plot(local_means.index, local_means[metric], 'o-')
    ax.set(xlabel='Input views', ylabel=metric, title='DTU: 22-scene mean of medians')
    ax.grid(alpha=0.3)
fig.suptitle('Figure 5 adaptation: uniform views, released HF weights')
fig.tight_layout()
fig.savefig(figure_dir / 'dtu_view_scaling.png', dpi=180)
plt.show()
for metric in ['accuracy_median', 'completion_median']:
    print(metric, '随视角数单调不增:', bool(np.all(np.diff(local_means[metric]) <= 0)))

<Figure size 1000x400 with 2 Axes>

accuracy_median 随视角数单调不增: True
completion_median 随视角数单调不增: False


### 步骤 6：解释性能测量（§4.1 / Table 2）

首个场景预热一次，再测量三次，记录模型内部分段计时以及包含 CPU 输出拷贝的 inference wall time。两种时间不是同一个概念。显存同时报告 PyTorch allocated 和 reserved；它们都不是 `nvidia-smi` 中含桌面与驱动开销的总占用。

论文 Table 2 是 A100 上 512×384 的输入；本实验是 RTX 5070 Ti Laptop 上的 512×512 DTU 图片，且 DPT head 分块为2。这里复现的是测量方法，不能拿绝对时间声称超越论文速度；也尚未运行 DUSt3R 对照。

In [5]:
timing_rows = []
for trial in ablation['performance']:
    assert trial['warmup'] == 1 and len(trial['samples']) == trial['repeats'] == 3
    timing_rows.append({'views': trial['view_count'],
        'forward_mean_s': np.mean([s['forward_seconds'] for s in trial['samples']]),
        'forward_std_s': np.std([s['forward_seconds'] for s in trial['samples']]),
        'wall_mean_s': np.mean([s['inference_wall_seconds'] for s in trial['samples']]),
        'peak_allocated_GiB': max(s['peak_allocated_gib'] for s in trial['samples']),
        'peak_reserved_GiB': max(s['peak_reserved_gib'] for s in trial['samples'])})
timing_df = pd.DataFrame(timing_rows).sort_values('views')
display(timing_df.round(4))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].errorbar(timing_df.views, timing_df.forward_mean_s, yerr=timing_df.forward_std_s, fmt='o-', capsize=4)
axes[0].set(xlabel='Input views', ylabel='Forward seconds (mean ± std)')
axes[1].plot(timing_df.views, timing_df.peak_allocated_GiB, 'o-', label='allocated')
axes[1].plot(timing_df.views, timing_df.peak_reserved_GiB, 's--', label='reserved')
axes[1].set(xlabel='Input views', ylabel='Peak GPU memory (GiB)')
axes[1].legend()
fig.suptitle('Table 2 methodology adaptation: RTX 5070 Ti Laptop, DPT chunk=2')
fig.tight_layout()
fig.savefig(figure_dir / 'inference_efficiency.png', dpi=180)
plt.show()

,views,forward_mean_s,forward_std_s,wall_mean_s,peak_allocated_GiB,peak_reserved_GiB
0,3,0.3108,0.0400,0.3189,4.4305,4.8730
1,5,0.5636,0.0659,0.5757,4.5649,5.1172
2,10,1.3030,0.0682,1.3254,4.9089,5.2969
3,20,2.9958,0.0083,3.0375,5.5827,5.9258


<Figure size 1000x400 with 2 Axes>

### 已生成的科研图

下面的 PNG 来自上述实测结果单元，GitHub 上也可直接查看：

![DTU view scaling](results/figures/dtu_view_scaling.png)

![Inference efficiency](results/figures/inference_efficiency.png)

### 已完成全量运行：Neural RGB-D（§4.3 / Table 3）

2026-10-01全量9场景评测正常退出，共278个采样视角；实际结果为 `results/nrgbd_seed42_stride40.json`。每40帧选一帧，使用完整有效轨迹，公开HF权重、seed42、512分辨率、aligned local点图、DPT chunk2。

Table 3 将距离乘100；加载器将毫米深度转为米，因此先在9场景间平均median距离再乘100。本地实测4.0165/1.2001，相比论文参考3.40/1.01分别高18.13%/18.82%，尚未匹配论文数值。公开权重与论文具体训练权重对应关系及原始下载revision未验证；chunk2是12GiB显存适配，不能认定其造成指标差距。

In [6]:
nrgbd_path = paper_root / 'results/nrgbd_seed42_stride40.json'
if not nrgbd_path.exists():
    print('Neural RGB-D：数据准备/评测尚未完成，当前没有本地指标。')
else:
    nrgbd_report = json.loads(nrgbd_path.read_text())
    assert nrgbd_report['dataset'] == 'nrgbd' and nrgbd_report['kf_every'] == 40
    assert nrgbd_report['seed'] == 42 and nrgbd_report['head'] == 'local'
    assert nrgbd_report['paper_distance_multiplier'] == 100
    nrgbd_manifest = json.loads((paper_root / 'results/nrgbd_data_manifest.json').read_text())
    assert nrgbd_report['scene_count'] == len(nrgbd_report['scenes']) == len(nrgbd_manifest['scene_names'])
    assert {r['scene'] for r in nrgbd_report['scenes']} == set(nrgbd_manifest['scene_names'])
    assert nrgbd_report['scene_count'] == len({r['scene'] for r in nrgbd_report['scenes']}) == 9
    for k, v in nrgbd_report['aggregate_mean'].items():
        values = [r['metrics'][k] for r in nrgbd_report['scenes']]
        assert np.isfinite(values).all() and np.isclose(v, np.mean(values), rtol=1e-12, atol=1e-12)
    nrgbd_comparison = pd.DataFrame([
        {'metric': 'Accuracy median', 'local x100': nrgbd_report['aggregate_mean']['accuracy_median'] * 100, 'paper Table 3': 3.40},
        {'metric': 'Completion median', 'local x100': nrgbd_report['aggregate_mean']['completion_median'] * 100, 'paper Table 3': 1.01},
    ])
    nrgbd_comparison['error increase %'] = (nrgbd_comparison['local x100'] / nrgbd_comparison['paper Table 3'] - 1) * 100
    display(nrgbd_comparison.round(4))
    nrgbd_scenes = pd.DataFrame([{'scene': r['scene'], 'views': r['views'],
        'Accuracy median x100': r['metrics']['accuracy_median'] * 100,
        'Completion median x100': r['metrics']['completion_median'] * 100}
        for r in nrgbd_report['scenes']])
    display(nrgbd_scenes.round(4))
    print(f"已核验 {len(nrgbd_scenes)} 个场景，共 {nrgbd_scenes['views'].sum()} 个采样视角；未匹配论文指标。")

,metric,local x100,paper Table 3,error increase %
0,Accuracy median,4.0165,3.40,18.1331
1,Completion median,1.2001,1.01,18.8220


,scene,views,Accuracy median x100,Completion median x100
0,breakfast_room,30,2.1250,0.4672
1,complete_kitchen,31,5.5303,1.9673
2,green_room,37,2.2289,0.4938
3,grey_white_room,38,7.5022,2.5886
4,kitchen,38,3.1274,1.2414
5,morning_apartment,23,1.0980,0.5411
6,staircase,29,5.0105,1.2555
7,thin_geometry,10,8.0700,1.8468
8,whiteroom,42,1.4563,0.3993


已核验 9 个场景，共 278 个采样视角；未匹配论文指标。


### 后续真实实验：配对点图消融与7-Scenes

当前归档：NRGBD全部9场景配对已完成。local/global的Accuracy mean×100为9.7108/9.2594，Completion mean×100为3.1292/3.1789；local提高normal consistency并降低Completion，但Accuracy没有改善，因此没有全面复现论文local优势。所有local指标与历史seed42/stride40运行逐项完全相等；这一重复验证不保证跨硬件逐bit一致。7-Scenes于2026-10-01 12:44:47（Asia/Shanghai）正常结束，覆盖全部7类场景、18条测试轨迹、850视角。Table3 median×100为3.3035/3.1058；Table5 mean×100为local6.3613/7.0516、global6.9567/6.3510，local不改善Completion，未匹配论文数值。

§5.4/Table 5：新增 `--head both`，同一输入、同一次网络预测依次测 aligned local 和 global，不能把独立随机前向的两个结果当作严格配对。选择metric点图不跳过local head计算，因此本实验不证明移除head的速度收益。下表使用mean距离，与Table 3/4的median不是同一指标。

§4.3/Table 3：7-Scenes由Microsoft官方TestSplit选择全部测试轨迹；仅存储原始stride20帧以节省磁盘，并在加载器中使用原始帧总数恢复完全相同的采样编号。非full_video或其他stride将报错。深度使用固定版本SimpleRecon的depth-to-RGB标定投影与z-buffer，不是TSDF重建深度，也不是把原始depth直接改名。保留测试序列ZIP的CRC/SHA256、源ETag与预处理源码SHA256；数据仅供非商业学术复现，不上传GitHub。

这里只在完整JSON存在且场景/轨迹集合和汇总指标通过核验后显示结果；运行中不以论文参考值填表。

![NRGBD paired head ablation](results/figures/nrgbd_paired_heads.png)

![7-Scenes paired head ablation](results/figures/7scenes_paired_heads.png)

In [7]:
paired_rows = []
for dataset_name, filename in [('nrgbd', 'nrgbd_paired_seed42_stride40.json'), ('7scenes', '7scenes_paired_seed42_stride20.json')]:
    path = paper_root / 'results' / filename
    if not path.exists():
        print(f'{dataset_name}：全量配对结果尚未生成。')
        continue
    report = json.loads(path.read_text())
    assert report['dataset'] == dataset_name and report['head'] == 'both' and report['paired_same_forward']
    assert report['aggregate_mean_head'] == 'local' and report['aggregate_mean'] == report['aggregate_by_head']['local']
    assert report['scene_count'] == len(report['scenes']) == len({r['scene'] for r in report['scenes']})
    if dataset_name == 'nrgbd':
        manifest = json.loads((paper_root / 'results/nrgbd_data_manifest.json').read_text())
        expected = set(manifest['scene_names'])
        assert len(expected) == 9
        old = json.loads((paper_root / 'results/nrgbd_seed42_stride40.json').read_text())
        old_rows = {r['scene']: r for r in old['scenes']}
        assert all(r['metrics_by_head']['local'] == old_rows[r['scene']]['metrics'] and r['seed'] == old_rows[r['scene']]['seed'] and r['views'] == old_rows[r['scene']]['views'] for r in report['scenes'])
    else:
        manifest = json.loads((paper_root / 'results/7scenes_data_manifest.json').read_text())
        assert manifest['status'] == 'prepared' and len(manifest['scenes_expected']) == 7
        expected = {r['scene'] + '/' + r['sequence'] for r in manifest['sequences']}
    assert {r['scene'] for r in report['scenes']} == expected
    for head in ['local', 'global']:
        for metric, value in report['aggregate_by_head'][head].items():
            values = [r['metrics_by_head'][head][metric] for r in report['scenes']]
            assert np.isfinite(values).all() and np.isclose(value, np.mean(values), rtol=1e-12, atol=1e-12)
        paired_rows.append({'dataset': dataset_name, 'head': head, 'trajectories': len(expected),
            'Accuracy mean x100': report['aggregate_by_head'][head]['accuracy'] * 100,
            'Completion mean x100': report['aggregate_by_head'][head]['completion'] * 100})
    assert all(r['paired_same_forward'] for r in report['scenes'])
if paired_rows:
    display(pd.DataFrame(paired_rows).round(4))
    print('这是Table 5 mean口径的本地配对实验，不是Table 3 median或整篇论文完成。')
nrgbd_paired_path = paper_root / 'results/nrgbd_paired_seed42_stride40.json'
if nrgbd_paired_path.exists():
    nrgbd_paired = json.loads(nrgbd_paired_path.read_text())
    deltas = pd.DataFrame([{'scene': r['scene'],
        'Accuracy': (r['metrics_by_head']['global']['accuracy'] - r['metrics_by_head']['local']['accuracy']) * 100,
        'Completion': (r['metrics_by_head']['global']['completion'] - r['metrics_by_head']['local']['completion']) * 100}
        for r in nrgbd_paired['scenes']]).set_index('scene')
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
    for ax, metric in zip(axes, ['Accuracy', 'Completion']):
        values = deltas[metric]
        ax.barh(deltas.index.str.replace('_', ' '), values, color=['#2563eb' if v < 0 else '#d97706' for v in values])
        ax.axvline(0, color='black', linewidth=1)
        ax.set_title(metric + ' mean distance x100')
        ax.set_xlabel('Global minus aligned local (negative: global better)')
        ax.grid(axis='x', alpha=.25)
    fig.suptitle('NRGBD paired head ablation: 9 scenes, same forward predictions')
    fig.tight_layout()
    fig.savefig(paper_root / 'results/figures/nrgbd_paired_heads.png', dpi=160, bbox_inches='tight')
    plt.show()

,dataset,head,trajectories,Accuracy mean x100,Completion mean x100
0,nrgbd,local,9,9.7108,3.1292
1,nrgbd,global,9,9.2594,3.1789
2,7scenes,local,18,6.3613,7.0516
3,7scenes,global,18,6.9567,6.3510


这是Table 5 mean口径的本地配对实验，不是Table 3 median或整篇论文完成。


<Figure size 1300x500 with 2 Axes>

In [8]:
seven_path = paper_root / 'results/7scenes_paired_seed42_stride20.json'
if not seven_path.exists():
    print('7-Scenes：完整评测尚未结束，暂不报告本地指标。')
    prepared_path = paper_root / 'results/7scenes_data_manifest.json'
    if prepared_path.exists():
        prepared = json.loads(prepared_path.read_text())
        if prepared['status'] == 'prepared':
            assert len(prepared['scenes_expected']) == 7 and prepared['trajectory_count'] == len(prepared['sequences']) == 18
            assert len({(r['scene'], r['sequence']) for r in prepared['sequences']}) == 18
            assert all(r['stored_indices'] == list(range(0, r['original_frame_count'], 20)) and r['selected_member_crc_checked'] and r['sequence_zip_crc_checked'] for r in prepared['sequences'])
            print(f"归档清单：7类场景、18测试轨迹、{sum(len(r['stored_indices']) for r in prepared['sequences'])}个stride20视角已准备。")
else:
    seven = json.loads(seven_path.read_text())
    from scripts.validate_7scenes_report import validate as validate_seven_report
    seven_validation = validate_seven_report(paper_root)
    print('完整集合、原始采样编号、seed、有限逐项aggregate及same-forward核验：', seven_validation['status'])
    assert seven['dataset'] == '7scenes' and seven['kf_every'] == 20 and seven['seed'] == 42
    assert seven['aggregate_mean_head'] == 'local' and seven['paper_distance_multiplier'] == 100
    seven_comparison = pd.DataFrame([
        {'metric': 'Accuracy median', 'local x100': seven['aggregate_mean']['accuracy_median'] * 100, 'paper Table 3': 1.58},
        {'metric': 'Completion median', 'local x100': seven['aggregate_mean']['completion_median'] * 100, 'paper Table 3': 0.93}])
    seven_comparison['error increase %'] = (seven_comparison['local x100'] / seven_comparison['paper Table 3'] - 1) * 100
    display(seven_comparison.round(4))
    print(f"7类场景、{seven['scene_count']}条测试轨迹、{seven_validation['sampled_views']}视角：先按官方实现逐轨迹median再平均、乘100。")
    seven_rows = pd.DataFrame([{'trajectory': r['scene'], 'views': r['views'],
        'Accuracy median x100': r['metrics_by_head']['local']['accuracy_median'] * 100,
        'Completion median x100': r['metrics_by_head']['local']['completion_median'] * 100}
        for r in seven['scenes']]).set_index('trajectory')
    display(seven_rows.round(4))
    deltas = pd.DataFrame([{'trajectory': r['scene'],
        'Accuracy': (r['metrics_by_head']['global']['accuracy'] - r['metrics_by_head']['local']['accuracy']) * 100,
        'Completion': (r['metrics_by_head']['global']['completion'] - r['metrics_by_head']['local']['completion']) * 100}
        for r in seven['scenes']]).set_index('trajectory')
    fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True)
    for ax, metric in zip(axes, ['Accuracy', 'Completion']):
        values = deltas[metric]
        ax.barh(deltas.index, values, color=['#2563eb' if v < 0 else '#d97706' for v in values])
        ax.axvline(0, color='black', linewidth=1)
        ax.set_title(metric + ' mean distance x100')
        ax.set_xlabel('Global minus aligned local (negative: global better)')
        ax.grid(axis='x', alpha=.25)
    fig.suptitle('7-Scenes paired head ablation: all 18 test trajectories, same forward')
    fig.tight_layout()
    fig.savefig(paper_root / 'results/figures/7scenes_paired_heads.png', dpi=160, bbox_inches='tight')
    plt.show()
    print('local改善汇总Accuracy，但Completion更差；公开权重与完整协议对应关系仍需核查，不能宣称整篇论文完成。')

完整集合、原始采样编号、seed、有限逐项aggregate及same-forward核验： passed


,metric,local x100,paper Table 3,error increase %
0,Accuracy median,3.3035,1.58,109.0794
1,Completion median,3.1058,0.93,233.9550


7类场景、18条测试轨迹、850视角：先按官方实现逐轨迹median再平均、乘100。


,views,Accuracy median x100,Completion median x100
trajectory,,,
chess/seq-03,50,0.4338,0.3685
chess/seq-05,50,1.0184,0.4722
fire/seq-03,50,0.2279,0.2514
fire/seq-04,50,0.1922,0.2228
heads/seq-01,50,1.3783,0.4870
office/seq-02,50,5.7091,1.8128
office/seq-06,50,17.9334,33.2884
office/seq-07,50,5.5626,3.5821
office/seq-09,50,5.7412,3.0902


<Figure size 1400x800 with 2 Axes>

local改善汇总Accuracy，但Completion更差；公开权重与完整协议对应关系仍需核查，不能宣称整篇论文完成。


### 哪些仍未完成？

1. §4.2/Table 1：CO3D 与 RealEstate10K 位姿 GT、划分与 RRA/RTA/mAA。PnP 输出或一张相机轨迹图不能替代这些 benchmark。
2. §4.3/Table 3/4：DTU22场景、Neural RGB-D9场景与7-Scenes全部18条测试轨迹已运行核验；论文数值仍未对齐，下一步做权重与协议差异审计。
3. §5.1 训练视角、§5.2 模型/数据 scaling、§5.3 无位置插值：需要各组独立训练模型。论文 §4 使用128张 A100-80GB，当前单卡无法按原规格训练。
4. 附录 C/D/E：Gaussian splatting、BA 与多视图深度；以及 §4.1 的大规模视角和 DUSt3R 基线。

合理的简历表述：基于官方公开权重搭建多视图重建流程，完成DTU22场景与Neural RGB-D9场景评测，记录论文指标差距，并设计视角规模与local/global点图消融。查看上面的实测表后再补入具体数字。

## 步骤6：公开权重与位姿数据预检查（§3.2、§4.2–4.3、§5.4）

三个重建数据集的实测已经保存，但整篇论文仍未完成。[协议审计](PROTOCOL_AUDIT.md)记录了当前公开HF权重的哈希匹配、公式与代码的置信度项符号差异、同像素RoMa注册与论文ICP文字的差异。它们是已核实事项，不是已证明的误差根因。

下面读取审计快照并再次检查位姿数据入口，不加载模型、不占用GPU、不下载数据。RealEstate10K规定文件实际包含1,832个唯一ID；预检查拒绝静默跳过缺失视频。CO3D需要可核验来源的selected_seqs_test.json。数据检查ready也只说明给定清单文件可读，不能证明论文协议一致。

上述是步骤6的历史资源快照；2026-10-02用户已扩容，下面步骤7更新数据准备状态。独立训练消融仍需不同训练模型的权重/算力。没有Table 1实测，不用PnP Demo填充论文成绩。

In [9]:
from pathlib import Path
import json
import sys
import pandas as pd

snapshot = json.loads(Path("results/protocol_audit_20261001.json").read_text())
checkpoint = snapshot["checkpoint"]
manifest = json.loads(Path("results/checkpoint_manifest.json").read_text())
for entry in manifest["files"]:
    key = "config" if entry["path"].endswith("config.json") else "weight"
    assert entry["sha256"] == checkpoint[f"local_{key}_sha256"]
    assert checkpoint[f"local_{key}_sha256"] == checkpoint[f"remote_{'lfs_' if key == 'weight' else ''}{key}_sha256"]
print("审计快照：", snapshot["checked_at_utc"])
print("公开HF文件哈希匹配；原始下载revision及论文实验权重映射仍未验证。")
print("当前公开revision：", checkpoint["current_remote_revision"])

sys.path.insert(0, str(Path("scripts").resolve()))
from check_pose_data import check_co3d, check_re10k
paths = snapshot["pose_data_preflight"]
co3d = check_co3d(paths["co3d_processed_root"])
re10k = check_re10k(paths["re10k_video_root"], paths["re10k_metadata_root"], paths["re10k_split_file"])
assert re10k["split_sha256"] == paths["re10k_split_sha256"]
print(pd.DataFrame([
    {"数据": "CO3D", "预检查": co3d["status"], "缺失/失败项": len(co3d["issues"])},
    {"数据": "RealEstate10K", "预检查": re10k["status"], "缺失/失败项": len(re10k["issues"])},
]).to_string(index=False))
print(f"RE10K指定{re10k['expected_scene_count']}个唯一视频，已准备{re10k['ready_scene_count']}个。")
print("10项合成预检查测试通过，不是RRA/RTA/mAA测评结果。")
print("本节不运行训练/位姿评测，整篇论文复现未完成。")

审计快照： 2026-10-01T05:07:25Z
公开HF文件哈希匹配；原始下载revision及论文实验权重映射仍未验证。
当前公开revision： a2c770b768ceb3a53c36c4f7a3619db0413dc3a1
           数据        预检查  缺失/失败项
         CO3D incomplete       1
RealEstate10K incomplete    1832
RE10K指定1832个唯一视频，已准备0个。
10项合成预检查测试通过，不是RRA/RTA/mAA测评结果。
本节不运行训练/位姿评测，整篇论文复现未完成。


## 步骤7：Table 1数据准备与可恢复位姿入口（2026-10-02）

用户已扩容，之前的磁盘限制解除。官方RE10K规定1832个相机TXT已完整准备，RGB归档由独立systemd服务下载；元数据齐备或归档下载完成不代表RGB/GT通过核验，更不代表位姿指标完成。

新单卡HF入口复用官方principal-point crop、focal/PnP与相对位姿指标，固定10视角采样，逐scene保存输入/协议SHA与真实poses。恢复时重新计算已有结果；缺场景不得静默跳过，PnP失败保留并计数。16-mixed、DPT chunk2、顺序固定OpenCV seed是本机适配，不冒充未公布论文采样。

下单元只验证相机记录及下载状态，不加载模型、不启动网络下载。[续接检查点](CONTINUATION.md)记录后台服务、日志、恢复命令与后续GT审计。额度不足时后台下载仍可运行；定时任务只能在后续触发且额度可用时尝试接续，不能保证重置瞬间恢复。

In [10]:
import hashlib
from check_pose_data import read_split, read_re10k_metadata
from prepare_re10k_metadata import checksum, EXPECTED_MD5
metadata_path = Path('results/re10k_metadata_manifest.json')
if not metadata_path.exists():
    print('官方相机元数据未完成；不填报Table 1分数。')
else:
    meta = json.loads(metadata_path.read_text())
    names = read_split('scripts/re10k_test_1800.txt')
    assert meta['status'] == 'metadata_prepared'
    assert meta['split_sha256'] == checksum('scripts/re10k_test_1800.txt')
    assert meta['archive_md5'] == EXPECTED_MD5
    assert set(meta['records']) == set(names)
    for name in names:
        path = Path(meta['metadata_root']) / (name + '.txt')
        assert checksum(path) == meta['records'][name]['sha256']
        assert len(read_re10k_metadata(path)) == meta['records'][name]['camera_record_count']
    print(f'官方相机TXT：{len(names)}/1832全部哈希与解析通过。')
    print('archive SHA256:', meta['archive_sha256'])
    partial = Path('data/re10k_test_only.zip.part')
    print('RGB归档当前字节：', partial.stat().st_size if partial.exists() else 0)
    print('RGB覆盖/GT与裁剪协议仍待审计；没有正式RRA/RTA/mAA分数。')
    print('新恢复入口的合成测试通过不等于完成真实GPU位姿评测。')
recovery_path = Path('results/re10k_rgb_recovery_20261002.json')
if recovery_path.exists():
    recovery = json.loads(recovery_path.read_text())
    assert recovery['expected_bytes'] - recovery['preserved_bytes'] == recovery['missing_bytes']
    assert not recovery['formal_pose_metrics_available']
    print('下载恢复历史检查点：', recovery['restart_time'], recovery['status'])
    print('保留字节/当时缺少字节：', recovery['preserved_bytes'], recovery['missing_bytes'])
    print('先重新传输并核对前缀，不代表下载完成；具体网络根因未确定。')
smoke_path = Path('results/re10k_smoke_seed42.json')
if smoke_path.exists():
    smoke = json.loads(smoke_path.read_text())
    assert smoke['protocol']['benchmark_scope'] == 'adaptation_smoke'
    assert smoke['scene_count'] == 1
    from fast3r_hf_re10k_pose_eval import aggregate, pose_metrics
    assert aggregate(smoke['scenes'], read_split('results/re10k_smoke_split.txt')) == smoke['aggregate_mean']
    for row in smoke['scenes']:
        recalculated, errors = pose_metrics(row['predicted_c2w'], row['gt_c2w'])
        assert recalculated == row['metrics']
        assert len(errors['rotation_deg']) == row['pair_count'] == 45
    print('真实GPU接线smoke：1场景/10视角/45对，PnP失败=', smoke['pnp_failed_view_count'])
    print(pd.Series(smoke['aggregate_mean']).to_string())
    print('以上只说明入口运行成功，不是Table1全量成绩，不说明达到了论文指标。')

官方相机TXT：1832/1832全部哈希与解析通过。
archive SHA256: af18e4d560ee9f73120128e53a56da215e118f3fb52018c14a66aa0f8ed691db
RGB归档当前字节： 54818855664
RGB覆盖/GT与裁剪协议仍待审计；没有正式RRA/RTA/mAA分数。
新恢复入口的合成测试通过不等于完成真实GPU位姿评测。
下载恢复历史检查点： 2026-10-02T03:52:39+08:00 recovery_started_not_download_complete
保留字节/当时缺少字节： 54818855664 786034185
先重新传输并核对前缀，不代表下载完成；具体网络根因未确定。


真实GPU接线smoke：1场景/10视角/45对，PnP失败= 0
RRA_at_5     0.555556
RRA_at_15    1.000000
RRA_at_30    1.000000
RTA_at_5     0.000000
RTA_at_15    0.088889
RTA_at_30    0.466667
mAA_30       0.162724
以上只说明入口运行成功，不是Table1全量成绩，不说明达到了论文指标。
